## 1. Download dos dados

**Objetivo**: Baixar todo historico de dados sobre os casos de dengue.

Fonte: API do InfoDengue, casos semanais de Teresina (2015 - 2026)

In [1]:
import pandas as pd
from pathlib import Path

# API do InfoDengue: Teresina (código IBGE 2211001), dengue, semanas de 2015 a 2026
link = "https://info.dengue.mat.br/api/alertcity?geocode=2211001&disease=dengue&format=csv&ew_start=1&ew_end=53&ey_start=2015&ey_end=2026"

CAMINHO_RAW = Path("../data/raw/infodengue_teresina.csv")

In [2]:
# Só baixa da API se ainda não tiver uma cópia salva.
# A API muda os números recentes com o tempo, com uma cópia fixa, os resultados ficam reproduzíveis.
# Pra atualizar os dados de propósito: apaga o arquivo e roda de novo.
if not CAMINHO_RAW.exists():
    pd.read_csv(link).to_csv(CAMINHO_RAW, index=False)

casos_de_dengue = pd.read_csv(CAMINHO_RAW)

## 2. Visão geral

Quantas linhas e colunas vieram, quais são e de que tipo.

In [3]:
# Verifica as colunas
casos_de_dengue.columns

Index(['data_iniSE', 'SE', 'casos_est', 'casos_est_min', 'casos_est_max',
       'casos', 'p_rt1', 'p_inc100k', 'Localidade_id', 'nivel', 'id',
       'versao_modelo', 'municipio_nome', 'Rt', 'pop', 'tempmin', 'umidmax',
       'receptivo', 'transmissao', 'nivel_inc', 'umidmed', 'umidmin',
       'tempmed', 'tempmax', 'casprov', 'casprov_est', 'casprov_est_min',
       'casprov_est_max', 'casconf', 'notif_accum_year'],
      dtype='str')

In [4]:
# Tipo de cada coluna
casos_de_dengue.dtypes

data_iniSE              str
SE                    int64
casos_est           float64
casos_est_min         int64
casos_est_max         int64
casos                 int64
p_rt1               float64
p_inc100k           float64
Localidade_id         int64
nivel                 int64
id                    int64
versao_modelo           str
municipio_nome          str
Rt                  float64
pop                   int64
tempmin             float64
umidmax             float64
receptivo           float64
transmissao           int64
nivel_inc             int64
umidmed             float64
umidmin             float64
tempmed             float64
tempmax             float64
casprov               int64
casprov_est         float64
casprov_est_min     float64
casprov_est_max     float64
casconf             float64
notif_accum_year      int64
dtype: object

In [5]:
# (linhas, colunas)
casos_de_dengue.shape

(615, 30)

## 3. Período e ordem

Conferindo se veio o período que eu pedi (2015 a 2026) e em que ordem as semanas chegam.

In [6]:
# Verifica o período que a API devolveu
casos_de_dengue["data_iniSE"].min(), casos_de_dengue["data_iniSE"].max()

('2015-01-04', '2026-10-11')

In [7]:
# 10 semanas mais recentes (a API devolve do mais novo pro mais antigo)
casos_de_dengue[["data_iniSE", "casos", "casos_est"]].head(10)

,data_iniSE,casos,casos_est
0,2026-10-11,14,48.0
1,2026-10-04,17,42.0
2,2026-09-27,20,39.0
3,2026-09-20,41,55.0
4,2026-09-13,30,41.0
5,2026-09-06,34,43.0
6,2026-08-30,52,59.0
7,2026-08-23,39,44.0
8,2026-08-16,60,63.0
9,2026-08-09,98,99.0


In [8]:
# 10 semanas mais antigas (2015)
casos_de_dengue[["data_iniSE", "casos", "casos_est"]].tail(10)

,data_iniSE,casos,casos_est
605,2015-03-08,280,280.0
606,2015-03-01,233,233.0
607,2015-02-22,137,137.0
608,2015-02-15,85,85.0
609,2015-02-08,75,75.0
610,2015-02-01,80,80.0
611,2015-01-25,46,46.0
612,2015-01-18,51,51.0
613,2015-01-11,34,34.0
614,2015-01-04,50,50.0


## 4. Valores faltando

Antes de decidir quais colunas usar, quero saber onde tem buraco.

In [9]:
# Quantos valores faltando po coluna
casos_de_dengue.isna().sum()

data_iniSE            0
SE                    0
casos_est             0
casos_est_min         0
casos_est_max         0
casos                 0
p_rt1                 0
p_inc100k             0
Localidade_id         0
nivel                 0
id                    0
versao_modelo         0
municipio_nome        0
Rt                    0
pop                   0
tempmin             442
umidmax             442
receptivo           313
transmissao           0
nivel_inc             0
umidmed             442
umidmin             442
tempmed             442
tempmax             442
casprov               0
casprov_est         615
casprov_est_min     615
casprov_est_max     615
casconf             615
notif_accum_year      0
dtype: int64

In [10]:
# Verifica a partir de quando as colunas de clima têm valor.
casos_de_dengue[casos_de_dengue["tempmed"].notna()]["data_iniSE"].min()

'2023-06-04'

## 5. Atraso de notificação

As semanas recentes têm bem menos `casos` do que `casos_est`. A ideia aqui é ver até quantas semanas pra trás essa diferença vai.

In [11]:
# Diferença entre o estimado e o notificado: mostra quantas semanas recentes ainda não "fecharam"
casos_de_dengue["diferenca"] = casos_de_dengue["casos_est"] - casos_de_dengue["casos"]
casos_de_dengue[["data_iniSE", "casos", "casos_est", "diferenca"]].head(20)

,data_iniSE,casos,casos_est,diferenca
0,2026-10-11,14,48.0,34.0
1,2026-10-04,17,42.0,25.0
2,2026-09-27,20,39.0,19.0
3,2026-09-20,41,55.0,14.0
4,2026-09-13,30,41.0,11.0
5,2026-09-06,34,43.0,9.0
6,2026-08-30,52,59.0,7.0
7,2026-08-23,39,44.0,5.0
8,2026-08-16,60,63.0,3.0
9,2026-08-09,98,99.0,1.0


## 6. Conclusões

- Dados baixados em 26/10/2026: 615 semanas e 30 colunas, de 04/01/2015 a 11/10/2026.
- A API devolve da semana mais recente pra mais antiga, e `data_iniSE` vem como texto (`str`). Na limpeza vou converter pra data e ordenar do mais antigo pro mais novo.
- Temperatura e umidade só têm valor a partir de junho de 2023, por isso os 442 vazios. Pra ter clima no período inteiro vou precisar dos dados do INMET.
- `casprov_est`, `casprov_est_min`, `casprov_est_max` e `casconf` estão 100% vazias, dá pra descartar. `receptivo` tem 313 vazios.
- Nas 10 semanas mais recentes, `casos` ainda é menor que `casos_est` (na última: 14 contra 48). Daí pra trás os dois batem. Por isso o alvo vai ser `casos_est`, e essas semanas ficam fora do treino e da avaliação.